## XGBoost v2

- Same model as XGBoost v1, but with more feature engineering
- Resulted in a 0.001 increase in accuracy

In [1]:
from dotenv import load_dotenv
import kagglehub
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
from xgboost import XGBClassifier

if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cuda'

In [2]:
load_dotenv()

if not Path("../Data/.complete/competitions/playground-series-s6e6/bundle.complete").is_file():
    kagglehub.competition_download("playground-series-s6e6", output_dir="Data")
    print("Data downloaded")
else:
    print("Data not downloaded, already exists")

Data not downloaded, already exists


In [3]:
train = pd.read_csv("../Data/train.csv", index_col="id")
train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
id,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [7]:
def add_features(df):
    df = df.copy()

    bands = ['u', 'g', 'r', 'i', 'z']

    # mag stats
    df['mag_mean'] = df[bands].mean(axis=1)
    df['mag_std'] = df[bands].std(axis=1)
    df['mag_max'] = df[bands].max(axis=1)
    df['mag_min'] = df[bands].min(axis=1)
    df['mag_range'] = df['mag_max'] - df['mag_min']

    # color indices
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['u_i'] = df['u'] - df['i']
    df['u_z'] = df['u'] - df['z']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['r_z'] = df['r'] - df['z']

    # spatial trigonometry
    alpha_rad = np.radians(df["alpha"])
    delta_rad = np.radians(df["delta"])
    df["alpha_sin"] = np.sin(alpha_rad)
    df["alpha_cos"] = np.cos(alpha_rad)
    df["delta_sin"] = np.sin(delta_rad)
    df["delta_cos"] = np.cos(delta_rad)

    # redshift interactions
    epsilon = 1e-5
    abs_redshift = df['redshift'].abs() + epsilon
    for col in ['u_g', 'g_r', 'r_i', 'i_z']:
        df[f'{col}_per_redshift'] = df[col] / abs_redshift
    for b in bands:
        df[f'redshift_{b}'] = df['redshift'] * df[b]
    
    return df

In [12]:
X = add_features(train).drop(columns=["class"])
categorical_cols = ["spectral_type", "galaxy_population"]
for col in categorical_cols:
    X[col] = X[col].astype("category")

le = LabelEncoder()
y = le.fit_transform(train["class"])

X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)
print(f'Features: {X_train.shape[1]}')

Features: 38


In [11]:
model = XGBClassifier(
    device=device,
    random_state=42,
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    early_stopping_rounds=100,
    enable_categorical=True
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)]
)

preds = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))
print(classification_report(y_val, preds, target_names=le.classes_))

[0]	validation_0-mlogloss:0.82030
[1]	validation_0-mlogloss:0.76877
[2]	validation_0-mlogloss:0.72330
[3]	validation_0-mlogloss:0.68276
[4]	validation_0-mlogloss:0.64616
[5]	validation_0-mlogloss:0.61288
[6]	validation_0-mlogloss:0.58234
[7]	validation_0-mlogloss:0.55446
[8]	validation_0-mlogloss:0.52871
[9]	validation_0-mlogloss:0.50495
[10]	validation_0-mlogloss:0.48298
[11]	validation_0-mlogloss:0.46258
[12]	validation_0-mlogloss:0.44360
[13]	validation_0-mlogloss:0.42582
[14]	validation_0-mlogloss:0.40927
[15]	validation_0-mlogloss:0.39379
[16]	validation_0-mlogloss:0.37930
[17]	validation_0-mlogloss:0.36574
[18]	validation_0-mlogloss:0.35293
[19]	validation_0-mlogloss:0.34087
[20]	validation_0-mlogloss:0.32964
[21]	validation_0-mlogloss:0.31884
[22]	validation_0-mlogloss:0.30887
[23]	validation_0-mlogloss:0.29932
[24]	validation_0-mlogloss:0.29032
[25]	validation_0-mlogloss:0.28189
[26]	validation_0-mlogloss:0.27379
[27]	validation_0-mlogloss:0.26625
[28]	validation_0-mlogloss:0.2

/home/jonanakai/miniconda3/envs/ds/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [13:17:10] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Accuracy: 0.9676972373776739
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98     75496
         QSO       0.97      0.96      0.96     23429
        STAR       0.93      0.92      0.93     16545

    accuracy                           0.97    115470
   macro avg       0.96      0.96      0.96    115470
weighted avg       0.97      0.97      0.97    115470



In [14]:
test = pd.read_csv("../Data/test.csv", index_col="id")

test = add_features(test)
for col in categorical_cols:
    test[col] = test[col].astype("category")

test_preds = model.predict(test)
test_pred_labels = le.inverse_transform(test_preds)

submission = pd.DataFrame({
    "id": test.index,
    "class": test_pred_labels
})

submission.to_csv("../Data/submission_xgboost_v2.csv", index=False)

submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
